In [1]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv('../../../../Downloads/otu-all.csv', index_col=0)

/var/folders/27/jj4ywyy10b175kd93kw7q3r00000gn/T/ipykernel_48913/826049186.py:1: DtypeWarning: Columns (157) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../../../../Downloads/otu-all.csv', index_col=0)


In [4]:
df.head()

,S1_bulk1,S1_bulk2,S1_bulk3,S1_bulk4,S1_bulk5,S1_rAo1,S1_rAo2,S1_rAo3,S1_rAo4,S1_rAo5,...,S6_rOmi7,S6_rOmi8,Unnamed: 150,Unnamed: 151,Unnamed: 152,Unnamed: 153,Unnamed: 154,Unnamed: 155,Unnamed: 156,Unnamed: 157
OTU_name,14390,14396,14397,14398,14399,14400,14401,14406,14407,14408,...,14970,14971,OTU_name,Kingdom,Phylum,Class,Order,Family,Genus,Species
OTU_2,847,2559,2157,3074,1659,411,2687,2703,3111,3148,...,241,56,OTU_2,Bacteria,Pseudomonadota,Alphaproteobacteria,Hyphomicrobiales,Xanthobacteraceae,Bradyrhizobium,NaN
OTU_3,55,134,8333,17,237,56,47,29,80,82,...,16,94,OTU_3,Bacteria,Bacillota,Bacilli,Bacillales,Bacillaceae,Bacillus,NaN
OTU_4,27,15,68,35,51,0,0,0,0,0,...,1077,488,OTU_4,Bacteria,Actinomycetota,Thermoleophilia,Solirubrobacterales,67-14,NaN,NaN
OTU_5,0,25,10,0,65,0,0,0,0,0,...,0,0,OTU_5,Bacteria,Actinomycetota,Actinobacteria,Pseudonocardiales,Pseudonocardiaceae,Crossiella,NaN


In [5]:
df = df.rename(index={'OTU_name': 'SampleID'})

In [6]:
samples = []
for t in ('S1_bulk', 'S2_bulk', 'S1_rAo', 'S2_rVo'):
    samples.extend([c for c in df.columns if t in c])
assert len(samples) == 40

In [7]:
# get raw counts
otu = df[samples]
otu.head()

,S1_bulk1,S1_bulk2,S1_bulk3,S1_bulk4,S1_bulk5,S1_bulk6,S1_bulk7,S1_bulk8,S1_bulk9,S1_bulk10,...,S2_rVo1,S2_rVo2,S2_rVo3,S2_rVo4,S2_rVo5,S2_rVo6,S2_rVo7,S2_rVo8,S2_rVo9,S2_rVo10
SampleID,14390,14396,14397,14398,14399,14922,14923,14924,14925,14926,...,14420,14421,14425,14426,14427,14428,14429,14936,14937,14938
OTU_2,847,2559,2157,3074,1659,734,3296,2374,896,1778,...,717,960,3356,1392,2129,433,776,743,2716,280
OTU_3,55,134,8333,17,237,15,21,0,0,859,...,135,227,797,5477,267,47,145,306,346,258
OTU_4,27,15,68,35,51,0,0,0,7,24,...,166,0,334,0,113,46,43,76,20,110
OTU_5,0,25,10,0,65,21,0,0,0,0,...,1352,451,55,21,128,83,161,646,253,112


In [8]:
mapping = pd.DataFrame(otu.iloc[0, :]).to_csv('../data/mapping.tsv', sep='\t')

In [9]:
otu = otu.iloc[1:, :]

In [10]:
otu = otu[~(otu == 0).all(axis=1)]
otu.to_csv('../data/otu_table.tsv', sep='\t')

In [13]:
tax_col = [i for i in df.columns if 'Unnamed: 15' in i]
tax = df[tax_col]
tax.columns = tax.iloc[0]
tax = tax[1:].reset_index(drop=True).\
    rename_axis(None, axis=1).set_index('OTU_name').rename_axis(None, axis=0)
tax = tax[tax.index.isin(otu.index)]
tax.head()

,Kingdom,Phylum,Class,Order,Family,Genus,Species
OTU_2,Bacteria,Pseudomonadota,Alphaproteobacteria,Hyphomicrobiales,Xanthobacteraceae,Bradyrhizobium,NaN
OTU_3,Bacteria,Bacillota,Bacilli,Bacillales,Bacillaceae,Bacillus,NaN
OTU_4,Bacteria,Actinomycetota,Thermoleophilia,Solirubrobacterales,67-14,NaN,NaN
OTU_5,Bacteria,Actinomycetota,Actinobacteria,Pseudonocardiales,Pseudonocardiaceae,Crossiella,NaN
OTU_7,Bacteria,Actinomycetota,Rubrobacteria,Rubrobacterales,Rubrobacteriaceae,Rubrobacter,NaN


In [14]:
tax.to_csv('../data/taxonomy.tsv', sep='\t')